# Moirai — Wind
Probabilistic rolling inference with optional dynamic NWP covariates.

This notebook reports the overall MAE, RMSE, and aQL result.


In [ ]:
# !pip install -r requirements/moirai.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "requirements").exists():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / "wind"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from preprocessing import load_wind, describe_wind, NWP_FEATURES, TEST_START, SEED, CONTEXT_LENGTH, PREDICTION_LENGTH, SITES
from src.metrics import print_metrics
from power_conversion import convert_wind_result_to_power

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
MODEL_QUANTILES=OUTPUT_QUANTILES
OUTPUT_DIR=ROOT/"outputs"/"predictions"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
data = load_wind(ROOT / "data" / "wind")
print(describe_wind(data))

from tqdm.auto import tqdm
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule


In [ ]:
VARIANT="with_covariates"
var_cols=NWP_FEATURES if VARIANT=="with_covariates" else []
model=MoiraiForecast(module=MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-base"),prediction_length=PREDICTION_LENGTH,context_length=CONTEXT_LENGTH,patch_size=8,num_samples=100,target_dim=1,feat_dynamic_real_dim=len(var_cols),past_feat_dynamic_real_dim=0)
predictor=model.create_predictor(batch_size=4)


In [ ]:
rolling={s:data.context_by_site[s].copy() for s in SITES}
frames=[]
for i in tqdm(range(len(data.daily_by_site[SITES[0]]))):
    day={s:data.daily_by_site[s][i][1].copy() for s in SITES}
    for site in SITES:
        context=rolling[site].sort_values("timestamp").iloc[-CONTEXT_LENGTH:]
        future=day[site].sort_values("timestamp")
        full=pd.concat([context,future],ignore_index=True)
        ds=PandasDataset.from_long_dataframe(full,target="target",item_id="item_id",timestamp="timestamp",freq="h",past_feat_dynamic_real=[],feat_dynamic_real=var_cols)
        _,template=split(ds,offset=-PREDICTION_LENGTH)
        instances=template.generate_instances(prediction_length=PREDICTION_LENGTH,windows=1,distance=PREDICTION_LENGTH)
        for fc in predictor.predict(instances.input):
            samples=fc.samples
            qhat={q:np.quantile(samples,q,axis=0) for q in MODEL_QUANTILES}
            ts=pd.date_range(fc.start_date.to_timestamp(),periods=PREDICTION_LENGTH,freq="h")
            frame=pd.DataFrame({"item_id":int(fc.item_id),"timestamp":ts,"prediction":qhat[0.5]})
            for q in MODEL_QUANTILES: frame[str(q)]=qhat[q]
            frames.append(frame)
    for site in SITES: rolling[site]=pd.concat([rolling[site],day[site]],ignore_index=True)
result=pd.concat(frames,ignore_index=True).merge(data.test[["item_id","timestamp","target"]],on=["item_id","timestamp"],how="inner")

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan
result.to_csv(OUTPUT_DIR / f"moirai_{VARIANT}.csv", index=False)
print_metrics(result, QUANTILES, label=f"moirai_{VARIANT}")

# Wind speed -> wind power evaluation
power_result=convert_wind_result_to_power(result)

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in power_result.columns:
        power_result[str(_q)] = np.nan
power_result.to_csv(OUTPUT_DIR/f"moirai_{VARIANT}_power.csv",index=False)
print_metrics(power_result,QUANTILES,label=f"moirai_{VARIANT} [wind power]")
